# Introduction: Time Series Features

In this notebook, we will look at different approaches for encoding time series variables. The objective is to determine the most effective method for representing date and time in a time series.

In [ ]:
import pandas as pd
import numpy as np

%load_ext autoreload
%autoreload 2

import sys
sys.path.append('..')

# Options for pandas
pd.options.display.max_columns = 20
pd.options.display.max_rows = 10

# Display all cell outputs
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

import plotly.plotly as py
import plotly.graph_objs as go
from plotly.offline import iplot, init_notebook_mode
init_notebook_mode(connected=True)

import cufflinks
cf.go_offline(connected=True)
cf.set_config_file(theme='pearl')


# Data: Building Energy Use

We'll work with building energy use at a 15 minute frequency. Building energy data is a useful time series because it exhibits patterns at the daily, weekly, and yearly levels. 

In [ ]:
data = pd.read_csv('data/building_1.csv', parse_dates=['timestamp'], usecols=['timestamp', 'energy'], index_col='timestamp').sort_index()
data.head()

In [ ]:
data.query('timestamp.dt.year == 2015').iplot(yTitle='Energy (Kwh)', title='Energy Use')

# Basic Time and Date Features

The first set of features we'll build are those we intuitively think of: minute, hour, dayofweek, month, dayofyear, year.

In [ ]:
baseline_features = ['minute', 'hour', 'dayofweek', 'day', 'week', 'month', 'dayofyear', 'year']

for feature in baseline_features:
    data['timestamp_' + feature] = getattr(data.index, feature)

data.head()

These are pretty straightforward. We should be careful to note the day of week goes from 0 (Monday) to 6 (Sunday).

In [ ]:
data.apply(lambda x: x.nunique(), axis=0)

# Fractional Features

If we want to compress these features, we can convert them into fractions of their respective interval. For example, fraction of the day, fraction of the month, fraction of the week, and fraction of the year,

In [ ]:
data['frac_day'] = ((data['timestamp_minute'] / 60) + data['timestamp_hour'])  / 24

data.query('timestamp.dt.month == 12 and timestamp.dt.year == 2015')['frac_day'].iplot(title='Fractional Day')

In [ ]:
data['frac_week'] = (data['frac_day'] + data['timestamp_dayofweek']) / 7

data.query('timestamp.dt.month == 12 and timestamp.dt.year == 2015')['frac_week'].iplot(title='Fractional Week')

For the fractional month, we have to use the number of `days_in_month` which is an attribute of a time series in Pandas.

In [ ]:
data['frac_month' ] = (data['timestamp_day'] + data['frac_day']) / data.index.days_in_month

data.query('timestamp.dt.year == 2015')['frac_month'].iplot(title='Fractional Month')

For the fraction of the year, we have to find the number of days in the year which obeys the rules:

1. 365 in a non-leap year
2. 366 in a leap year which occurs:
    * When the year is divisible by 4 except:
        * Years divisible by 100 and not divisible by 400
        
This logic is captured below

In [ ]:
def find_days_in_year(years):
    return np.where((years % 4 == 0) & ((years % 100 != 0) | (years % 400 == 0)), 366, 365)


find_days_in_year(np.array([1900, 2000, 2004, 2005, 2009, 2010, 3000, 3200]))

In [ ]:
days_in_year = find_days_in_year(data['timestamp_year'])
np.unique(days_in_year)

In [ ]:
data['frac_year'] = (data['timestamp_dayofyear'] + data['frac_day']) / days_in_year

data.query('timestamp.dt.year == 2015')['frac_year'].iplot(title='Fractional Year')